# Sachsen-Anhalt 2026: full Git history and independent comparison

Frozen endpoint: 80fa3052044a45af29f4f0b2867957d8a3b1df35. This notebook independently parses original Git CSV/HTML, checks all final aggregates, verifies the named historical cases, and recalculates the external robust screening from original values. No live election polling or publication. The report contains 25 German tweet drafts and 25 regenerated charts.


In [1]:
from pathlib import Path
import csv, io, json, subprocess, hashlib, re, gzip, statistics, math
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/analyze_lsa_git_timeline.py').exists())
REF = '80fa3052044a45af29f4f0b2867957d8a3b1df35'
OUT = ROOT / 'data/2026-lsa/reports/git-timeline' / REF[:8]
BASE = 'data/2026-lsa/latest/'
summary = json.loads((OUT/'summary.json').read_text())
assert summary['ref']==REF
assert summary['reporting_evidence']['complete_reconciled']
def git_file(ref,path):
    return subprocess.check_output(['git','show',f'{ref}:{path}'],cwd=ROOT)
source_cache={}
def official(ref):
    if ref not in source_cache:
        manifest=json.loads(git_file(ref,BASE+'official_sources/manifest.json'))
        records=[]
        for entry in manifest['fetches']:
            if entry['filename'].endswith('.csv'):
                data=git_file(ref,BASE+'official_sources/'+entry['filename'])
                assert hashlib.sha256(data).hexdigest()==entry['content_hash']
                records.extend(csv.DictReader(io.StringIO(data.decode('utf-8-sig')),delimiter=';'))
        source_cache[ref]=records
    return source_cache[ref]
def integer(v):return int(v) if v not in ('',None) else None
raw=official(REF)
land=next(x for x in raw if x.get('Satzart')=='LAN' and not x['Wahllokal'])
booths=[x for x in raw if 'Wahlbezirk' in x]
assert len(booths)==2661
assert 'Ist.Wahlbezirke' not in land and 'Soll.Wahlbezirke' not in land
print({'data_commits':summary['history_commit_count'],'dated_captures':summary['versions'],'booths':len(booths),'valid_second_votes':integer(land['F.Gültige.Zweitstimmen'])})


{'data_commits': 124, 'dated_captures': 122, 'booths': 2661, 'valid_second_votes': 1315315}


## Independent final arithmetic and aggregation

Each geographic level is summed separately. Candidate blanks are preserved as missing in source data; the sum of present candidates is compared with the valid total. Postal A=0 is not used as a turnout denominator. Dropped Land counter columns are not assigned synthetic values.


In [2]:
fields=['B.Wähler','D.Gültige.Erststimmen','F.Gültige.Zweitstimmen']+[f for f in land if re.fullmatch(r'[DF]\d+\..+',f)]
checks=0
for kind,expected in [('KRS',14),('WKR',41),('GEM',218),('WBZ',2661)]:
    children=booths if kind=='WBZ' else [x for x in raw if x.get('Satzart')==kind and not x['Wahllokal']]
    assert len(children)==expected
    for f in fields:
        assert sum(integer(x.get(f)) or 0 for x in children)==integer(land[f]),(kind,f)
        checks+=1
for parent in [x for x in raw if x.get('Satzart') in ('GEM','WKR') and not x['Wahllokal']]:
    child=[x for x in booths if x['Gemeindeschlüssel']==parent['Schlüsselnummer']] if parent['Satzart']=='GEM' else [x for x in booths if int(x['Wahlkreisnummer'])==int(parent['Schlüsselnummer'])]
    assert child
    for f in fields:
        assert sum(integer(x.get(f)) or 0 for x in child)==(integer(parent.get(f)) or 0),(parent['Name'],f)
        checks+=1
arithmetic=0
for x in raw:
    if 'B.Wähler' not in x:continue
    for prefix,valid,invalid in [('D','D.Gültige.Erststimmen','C.Ungültige.Erststimmen'),('F','F.Gültige.Zweitstimmen','E.Ungültige.Zweitstimmen')]:
        assert sum(integer(v) or 0 for k,v in x.items() if re.fullmatch(prefix+r'\d+\..+',k))==integer(x[valid])
        assert integer(x[valid])+integer(x[invalid])==integer(x['B.Wähler'])
        arithmetic+=2
print({'independent_geographic_fields':checks,'independent_ballot_and_party_checks':arithmetic,'discrepancies':0})


{'independent_geographic_fields': 8942, 'independent_ballot_and_party_checks': 13932, 'discrepancies': 0}


In [3]:
gap=json.loads((OUT/'aggregation_gap.json').read_text())
first=gap['first_capture']['commit'];prior=gap['matching_municipality_arrival']['previous_commit']
a=next(x for x in official(prior) if x.get('Satzart')=='GEM' and x['Schlüsselnummer']=='15089015' and not x['Wahllokal'])
b=next(x for x in official(first) if x.get('Satzart')=='GEM' and x['Schlüsselnummer']=='15089015' and not x['Wahllokal'])
booth=next(x for x in booths if x['Gemeindeschlüssel']=='15089015' and x['Wahlbezirk']=='000965')
for f in fields:
    assert (integer(b.get(f)) or 0)-(integer(a.get(f)) or 0)==(integer(booth.get(f)) or 0),f
for ref in gap['observed_commits']:
    rr=official(ref);ll=next(x for x in rr if x.get('Satzart')=='LAN' and not x['Wahllokal']);gg=[x for x in rr if x.get('Satzart')=='GEM' and not x['Wahllokal']]
    assert sum(integer(x['F.Gültige.Zweitstimmen']) for x in gg)-integer(ll['F.Gültige.Zweitstimmen'])==1513
aken=next(x for x in booths if x['Gemeindeschlüssel']=='15082005' and x['Wahlbezirk']=='000010')
assert integer(aken['F.Gültige.Zweitstimmen'])==1139
assert 1139 != next(e for e in summary['denominator_changes'] if e['level']=='GEMEINDE')['changes']['valid_votes_zweit']['delta']
print('Aschersleben: all party and voter deltas match final 000965; three historical 1513-vote gaps, final gap zero.')
print('Aken: final individual second votes 1139; historical municipality arrival 435. Not interchangeable.')


Aschersleben: all party and voter deltas match final 000965; three historical 1513-vote gaps, final gap zero.
Aken: final individual second votes 1139; historical municipality arrival 435. Not interchangeable.


In [4]:
def status_table(ref):
    html=git_file(ref,BASE+'official_sources/overview.html').decode('utf-8-sig')
    for attrs,body in re.findall(r'<script([^>]*)>(.*?)</script>',html,re.S|re.I):
        if 'application/json' not in attrs.lower():continue
        data=json.loads(body).get('x',{}).get('tag',{}).get('attribs',{}).get('data',{})
        if isinstance(data,dict) and 'wbz_ist' in data:
            return [{k:v[i] for k,v in data.items() if isinstance(v,list) and len(v)==len(data['wbz_ist'])} for i in range(len(data['wbz_ist']))]
    raise AssertionError('Missing status table')
loss=summary['status_losses'][0]
for ref,expected in [(loss['commit'],0),(loss['restored_capture']['commit'],1)]:
    records=status_table(ref)
    target=next(x for x in records if x['Gemeinde']=='Bitterfeld-Wolfen, Stadt' and x['wbz']=='000028')
    assert int(target['wbz_ist'])==expected
bf=next(x for x in booths if x['Gemeindeschlüssel']=='15082015' and x['Wahlbezirk']=='000028')
assert integer(bf['F.Gültige.Zweitstimmen'])==1024
final=status_table(REF)
status_keys={(int(x['Wahlkreis'].split(',')[0]),x['Gemeinde'],x['wbz']) for x in final}
vote_keys={(int(x['Wahlkreisnummer']),x['Gemeindename'],x['Wahlbezirk']) for x in booths}
assert status_keys==vote_keys and len(status_keys)==2661
assert all(int(x['wbz_ist'])==int(x['wbz_soll'])==1 for x in final)
print('Bitterfeld 000028: historical statuses 0 -> 1; final 1024 second votes, historical vote diff unavailable.')
print('Every final HTML identity matches an original individual vote CSV identity: 2661 / 2661.')


Bitterfeld 000028: historical statuses 0 -> 1; final 1024 second votes, historical vote diff unavailable.
Every final HTML identity matches an original individual vote CSV identity: 2661 / 2661.


## Independent comparison with the frozen external analysis

The linked analysis is a secondary analytical source, not an instruction. All data here comes from its archived, versioned public export. Its robust statistic is independently recomputed directly from the original German CSV columns. Threshold colors are descriptive screening rules; there is no calibrated election-error probability.


In [5]:
EXT=OUT/'external'
def external(name):
    obj=json.loads((EXT/(name+'.json')).read_text())
    if isinstance(obj,dict) and obj.get('format')=='wahlb-json-parts-v1':
        return [r for part in obj['parts'] for r in json.loads((EXT/part).read_text())]
    return obj
for item in json.loads((EXT/'manifest.json').read_text()):
    assert hashlib.sha256((EXT/item['file']).read_bytes()).hexdigest()==item['sha256']
def raw_key(x):
    if 'Wahlbezirk' in x:return ':'.join(['WBZ',x['Wahlkreisnummer'],x['Kreisschlüssel'],x['Gemeindeschlüssel'],x['Wahlbezirk'],x['Wahllokal']])
    return ':'.join([x['Satzart'],x['Schlüsselnummer'],x['Wahllokal'] or 'TOTAL'])
lookup={raw_key(x):x for x in raw if 'B.Wähler' in x}
groups={}
for k,x in lookup.items():
    if k.startswith('WBZ:') or (k.startswith('GEM:') and k.endswith(':TOTAL')):
        if integer(x['B.Wähler'])>=100:
            group=(k.split(':')[0],x['Wahllokal'] or 'TOTAL',int(math.log10(integer(x['B.Wähler']))))
            groups.setdefault(group,[]).append(x)
def value(x,f):
    B=integer(x['B.Wähler'])
    if f=='Beteiligung':return 100*B/integer(x['A.Wahlberechtigte']) if integer(x['A.Wahlberechtigte']) and x['Wahllokal']!='B' else None
    if f=='Ungültige Erststimmen':return 100*integer(x['C.Ungültige.Erststimmen'])/B
    if f=='Ungültige Zweitstimmen':return 100*integer(x['E.Ungültige.Zweitstimmen'])/B
    col=next(k for k in x if re.fullmatch(r'F\d+\.'+re.escape(f),k))
    return 100*integer(x[col])/integer(x['F.Gültige.Zweitstimmen'])
cache={};checked=0;flag_counts={'orange':0,'red':0};areas=set()
for e in external('method-robust-mad'):
    if e['status']!='checked':continue
    x=lookup[e['row']];g=(e['row'].split(':')[0],x['Wahllokal'] or 'TOTAL',int(math.log10(integer(x['B.Wähler']))));f=e['field']
    if (g,f) not in cache:
        vals=[value(y,f) for y in groups[g]];vals=[z for z in vals if z is not None]
        median=statistics.median(vals);mad=statistics.median(abs(z-median) for z in vals)
        cache[g,f]=(median,mad,len(vals))
    med,mad,count=cache[g,f];obs=value(x,f);z=(obs-med)/(1.4826*mad)
    assert all(math.isclose(a,b,abs_tol=1e-9) for a,b in [(obs,e['observed']),(med,e['expected']),(mad,e['mad']),(z,e['z'])]) and count==e['n']
    level='red' if abs(z)>=10 and abs(obs-med)>=5 else 'orange' if abs(z)>=6 and abs(obs-med)>=2 else 'neutral'
    assert level==e['level'];checked+=1
    if level!='neutral':flag_counts[level]+=1;areas.add(e['row'])
assert checked==24171 and flag_counts=={'orange':88,'red':30} and len(areas)==106
print({'checked_MAD_values':checked,'flags':flag_counts,'distinct_areas':len(areas),'differences':0})


{'checked_MAD_values': 24171, 'flags': {'orange': 88, 'red': 30}, 'distinct_areas': 106, 'differences': 0}


In [6]:
mapping={'Wähler':'B.Wähler','Wahlberechtigte':'A.Wahlberechtigte','Gültige Erststimmen':'D.Gültige.Erststimmen','Gültige Zweitstimmen':'F.Gültige.Zweitstimmen','Ungültige Erststimmen':'C.Ungültige.Erststimmen','Ungültige Zweitstimmen':'E.Ungültige.Zweitstimmen','Gemeldete Wahlbezirke':'Ist.Wahlbezirke','Vorgesehene Wahlbezirke':'Soll.Wahlbezirke','Erwartete Wahlbezirke':'Soll.Wahlbezirke'}
matched=json.loads((OUT/'external_revision_comparison.json').read_text())
lookups={}
verified=0
for e in matched:
    match=e['exact_git_matches'][0];new_ref=match['commit']
    with (OUT/'raw_candidate_events.csv').open() as f:
        event=next(x for x in csv.DictReader(f) if x['commit']==new_ref and x['key'].endswith(':'+e['external_row'].split(':')[-1]) and e['mapped_field'] in json.loads(x['changes']) and json.loads(x['changes'])[e['mapped_field']]['before']==e['before'] and json.loads(x['changes'])[e['mapped_field']]['after']==e['after'])
    # Use the matched row's exact identity; the source pair comes from the audit's event record.
    for ref in [new_ref,event['previous_commit']]:
        if ref not in lookups:lookups[ref]={raw_key(x):x for x in official(ref) if 'B.Wähler' in x}
    old=lookups[event['previous_commit']][e['external_row']];new=lookups[new_ref][e['external_row']]
    field=mapping.get(e['field'])
    if field is None:
        prefix,name=e['field'].split(': ',1)
        field=next(k for k in new if re.fullmatch(('D' if prefix=='Erststimmen' else 'F')+r'\d+\.'+re.escape(name),k))
    assert integer(old[field])==e['before'] and integer(new[field])==e['after']
    verified+=1
assert verified==86
print({'external_revision_fields_verified_against_original_Git_CSV':verified})


{'external_revision_fields_verified_against_original_Git_CSV': 86}


In [7]:
all_commits=set(subprocess.check_output(['git','log','--format=%H',REF,'--',BASE],cwd=ROOT).decode().splitlines())
first_parent=set(subprocess.check_output(['git','log','--first-parent','--format=%H',REF,'--',BASE],cwd=ROOT).decode().splitlines())
assert all_commits==first_parent and len(all_commits)==124
assert len(summary['municipality_revisions'])==20
assert len(summary['status_losses'])==1 and len(summary['status_identity_additions'])==1
assert not summary['arithmetic_issues'] and not summary['source_replay_differences']
print('Full reachable data history equals first-parent scope: 124 commits. Historic events remain in the report.')
print('Limits: one individual-vote snapshot; no historical district party diff. External simulations and spatial/multivariate models were not rerun.')


Full reachable data history equals first-parent scope: 124 commits. Historic events remain in the report.
Limits: one individual-vote snapshot; no historical district party diff. External simulations and spatial/multivariate models were not rerun.


In [8]:
from fractions import Fraction
with (OUT/'cross_election_share_delta.csv').open() as f:comparison=list(csv.DictReader(f))
for x in comparison:
    if not x['relative_delta_percent']:continue
    first=Fraction(int(x['first_votes']),int(x['first_valid_votes']))
    last=Fraction(int(x['last_votes']),int(x['last_valid_votes']))
    expected=100*(last-first)/first
    assert math.isclose(float(expected),float(x['relative_delta_percent']),abs_tol=1e-10)
assert len(comparison)==18
with (OUT/'afd_arrival_observations.csv').open() as f:arrivals=list(csv.DictReader(f))
for kind in ['first_positive_result','first_complete_result']:
    subset=[x for x in arrivals if x['arrival_definition']==kind]
    assert len(subset)==len({x['key'] for x in subset})==273
    assert sum(x['level']=='GEMEINDE' for x in subset)==218
    assert sum(x['level']=='WAHLKREIS' for x in subset)==41
    assert sum(x['level']=='KREIS' for x in subset)==14
    for x in subset:assert math.isclose(float(x['afd_share']),100*int(x['afd_votes'])/int(x['valid_votes_zweit']))
with (OUT/'statla_second_vote_representation_waterfall.csv').open() as f:bridge=list(csv.DictReader(f))
assert int(bridge[-1]['end'])==0
for x in bridge:
    assert int(x['start'])-int(x['amount'])==int(x['end']) if x['type']=='delta' else int(x['end'])==int(x['amount'])
representation=json.loads((OUT/'representation_sources.json').read_text())
assert representation['represented_votes']==1224292
assert representation['represented_votes']+representation['not_represented_valid_votes']==1315315
assert hashlib.sha256((OUT/representation['population_file']).read_bytes()).hexdigest()==representation['population_sha256']
print('18 relative changes checked with exact fractions; 546 arrival observations verified; representation bridge closes at zero.')
print('RLP early FREIE WÄHLER parser defect is disclosed; its party is outside the six-party chart and its zero is not imputed.')


18 relative changes checked with exact fractions; 546 arrival observations verified; representation bridge closes at zero.
RLP early FREIE WÄHLER parser defect is disclosed; its party is outside the six-party chart and its zero is not imputed.
